# Day 5 Project Solution: Hardened safe_ask

All three engineering hygiene layers: logging, error handling, and tests.

## Setup

In [ ]:
import logging
import ollama

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)
logger = logging.getLogger(__name__)

MODEL = "llama3.2"
SYSTEM = "You are a helpful assistant. Answer concisely."

## Implementation

In [ ]:
def safe_ask(question: str, fallback: str = "") -> str:
    """Ask the local LLM. Returns fallback if Ollama is unavailable."""
    logger.debug("safe_ask() | question=%r", question[:60])
    try:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user",   "content": question},
            ],
        )
        answer = response["message"]["content"]
        logger.info("safe_ask() | got %d chars", len(answer))
        return answer
    except Exception as e:
        logger.error("safe_ask() failed: %s", e)
        if fallback:
            return fallback
        raise

## Tests

In [ ]:
def test_safe_ask_returns_string():
    result = safe_ask("Say 'hello' in one word.")
    assert isinstance(result, str) and len(result) > 0, \
        f'expected non-empty str, got {result!r}'


def test_safe_ask_fallback_on_bad_model():
    global MODEL
    _orig = MODEL
    MODEL = "no-such-model-xyz-day005"
    try:
        result = safe_ask("hello", fallback="unavailable")
        assert result == "unavailable", f'expected "unavailable", got {result!r}'
    finally:
        MODEL = _orig

## Run Tests

In [ ]:
# Run tests
test_safe_ask_returns_string()
print("✅ test_safe_ask_returns_string passed")

test_safe_ask_fallback_on_bad_model()
print("✅ test_safe_ask_fallback_on_bad_model passed")

## Gate Check

In [ ]:
# Gate check — verify Ollama is reachable
import urllib.request
try:
    urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
except Exception as e:
    raise AssertionError(f"Ollama server is not running: {e}") from e

print("✅ Day 5 gate: Ollama running, safe_ask implemented, tests pass.")